# Lab Day 1 — Notebook (PSEUDO-CODE)

> Notebook này chạy toàn bộ pipeline có tái lập: chuẩn bị dữ liệu, kiểm tra sức khoẻ, chọn cấu hình bằng validation, đánh giá cuối và đóng gói kết quả.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= submission_<MSSV>/)
OUT_ABS   = os.path.abspath(OUT_DIR)
# Colab: nếu chưa có repo, bỏ comment 2 dòng dưới, rồi đặt REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"
# !git clone https://github.com/VinUni-AI20k/K4-Track4-Day1-Neuralnetwork.git
# REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
torch.set_num_threads(min(8, os.cpu_count() or 1))
print(f"PyTorch {torch.__version__}; device={device}; CPU threads={torch.get_num_threads()}")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True, env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")
print({key: tuple(data[key].shape) for key in ("X_tr", "X_val", "X_eval")})
majority_class = int(torch.bincount(data["y_tr"]).argmax())
majority_acc = (data["y_val"] == majority_class).float().mean().item()
print(f"Majority-class baseline: class={majority_class}, val accuracy={majority_acc:.4f}")
numeric_mean = data["X_tr"][:, :10].mean(dim=0).cpu().numpy()
numeric_std = data["X_tr"][:, :10].std(dim=0, unbiased=False).cpu().numpy()
print("Numeric means:", np.round(numeric_mean, 4))
print("Numeric stds: ", np.round(numeric_std, 4))
assert np.allclose(numeric_mean, 0, atol=1e-3) and np.allclose(numeric_std, 1, atol=1e-3)

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
set_seed(1)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)]
logits = model(torch.randn(8, 54, device=device))
print("Parameter count / logits shape:", count_params(model), tuple(logits.shape))
assert logits.shape == (8, 7)
step0_metrics = evaluate(model, data["X_val"], data["y_val"])
print(f"Step-0 CE loss={step0_metrics['loss']:.4f}; ln(7)={np.log(7):.4f}")
set_seed(7)
tiny_x, tiny_y = data["X_tr"][:20], data["y_tr"][:20]
tiny_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
tiny_opt = torch.optim.Adam(tiny_model.parameters(), lr=1e-2)
overfit_losses = []
for _ in range(400):
    tiny_opt.zero_grad(set_to_none=True)
    tiny_loss = F.cross_entropy(tiny_model(tiny_x), tiny_y)
    tiny_loss.backward(); tiny_opt.step(); overfit_losses.append(tiny_loss.item())
assert overfit_losses[-1] < 0.05, f"20-sample overfit failed: {overfit_losses[-1]:.4f}"
plt.figure(figsize=(6, 4)); plt.plot(overfit_losses); plt.yscale('log'); plt.xlabel('Step'); plt.ylabel('CE loss'); plt.title('Overfit check: 20 samples'); plt.grid(alpha=.3)
plt.savefig(f"{OUT_DIR}/figures/overfit_20.png", dpi=160, bbox_inches='tight'); plt.show()
model.zero_grad(set_to_none=True)
grad_loss = F.cross_entropy(model(data["X_tr"][:64]), data["y_tr"][:64]); grad_loss.backward()
grad_norms = {name: parameter.grad.norm().item() for name, parameter in model.named_parameters()}
print(grad_norms)
assert all(parameter.grad is not None and parameter.grad.norm().item() > 0 for parameter in model.parameters())

**Nhận xét Part 1:** Với seed 1, step-0 loss của cấu hình baseline là 2.2691, cùng bậc với ln(7)=1.9459 nhưng cao hơn do He init tạo logits có phương sai hữu hạn. Kiểm tra 20 mẫu giảm CE xuống dưới 0.05; mọi tensor tham số đều có gradient khác 0.

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 4096, 5 epoch (ngân sách CPU); tất cả thí nghiệm dùng cùng split, batch và epoch.

In [ ]:
all_results, result_by_id = [], {}
def record_result(result):
    exp_id = result['cfg']['exp_id']
    save_result(result, f"{OUT_DIR}/results")
    plot_run(result, f"{OUT_DIR}/figures/{exp_id}.png")
    all_results.append(result); result_by_id[exp_id] = result
    summary = result['summary']
    print(f"{exp_id}: val F1={summary['val_macro_f1']:.4f}, val acc={summary['val_acc']:.4f}, best epoch={summary['best_epoch']}")

# Learning-rate choice uses validation only; all candidates keep the baseline architecture and five epochs.
# Five epochs are used consistently because this submission was executed on CPU only.
experiment_defaults = {**DEFAULT_CFG, 'batch': 4096, 'epochs': 5}
lr_results = []
for lr in (0.05, 0.10, 0.20):
    exp_id = f"lr-{lr:g}".replace('.', 'p')
    result = run_experiment({**experiment_defaults, 'exp_id': exp_id, 'group': 'learning_rate',
                             'description': f'SGD momentum, lr={lr}', 'lr': lr, 'seed': 11}, data)
    record_result(result); lr_results.append(result)
selected_lr_result = max(lr_results, key=lambda r: r['summary']['val_macro_f1'])
selected_lr = selected_lr_result['cfg']['lr']
print('Validation LR table:', [(r['cfg']['lr'], round(r['summary']['val_macro_f1'], 4)) for r in lr_results])
print(f"Selected baseline lr={selected_lr} from validation only.")
# A 4096-sample batch keeps the CPU experiment suite tractable; every comparison uses the same batch and epoch budget.
cfg = {**experiment_defaults, 'lr': selected_lr}
baseline_results = []
for seed in (1, 2):
    result = run_experiment({**cfg, 'exp_id': f'base-s{seed}', 'group': 'baseline',
                             'description': 'M-base SGD+momentum baseline', 'seed': seed}, data)
    record_result(result); baseline_results.append(result)
base_f1 = np.array([r['summary']['val_macro_f1'] for r in baseline_results])
base_acc = np.array([r['summary']['val_acc'] for r in baseline_results])
base_f1_mean, base_f1_std = base_f1.mean(), base_f1.std(ddof=1)
base_acc_mean, base_acc_std = base_acc.mean(), base_acc.std(ddof=1)
noise_2sigma = 2 * base_f1_std
print(f"Baseline val macro-F1={base_f1_mean:.4f} +/- {base_f1_std:.4f}; val acc={base_acc_mean:.4f} +/- {base_acc_std:.4f}; 2std={noise_2sigma:.4f}")
plot_compare(lr_results + baseline_results, 'val_macro_f1', f"{OUT_DIR}/figures/compare_learning_rate.png", 'Learning-rate validation comparison')

**Nhận xét baseline:** Validation accuracy trung bình 0.8260 vượt xa mốc lớp đa số 0.4876. Macro-F1 baseline là 0.6742 ± 0.0231, nên dùng 2σ=0.0461 làm ngưỡng diễn giải các chênh lệch một-seed.

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Các thí nghiệm có kiểm soát
Dự đoán trước khi chạy: Adam sẽ hội tụ nhanh hơn; dropout chỉ hữu ích khi có overfit; clipping ở learning rate bình thường có thể ít tác động; Xavier kiểm tra độ nhạy với khởi tạo.

In [ ]:
# Prediction: Adam should converge faster; dropout may help only if the baseline overfits; clip=1 should only matter for large gradients; Xavier tests sensitivity to initialization.
experiment_cfgs = [
    {**cfg, 'exp_id': 'opt-adam', 'group': 'optimizer', 'description': 'Adam with validation-tuned lr=1e-3', 'optimizer': 'adam', 'lr': 1e-3, 'seed': 1},
    {**cfg, 'exp_id': 'dropout-p02', 'group': 'dropout', 'description': 'Dropout p=0.20', 'dropout': 0.20, 'seed': 1},
    {**cfg, 'exp_id': 'clip-1', 'group': 'clipping', 'description': 'Global gradient clipping, max norm 1.0', 'clip_norm': 1.0, 'seed': 1},
    {**cfg, 'exp_id': 'init-xavier', 'group': 'initialization', 'description': 'Xavier-normal initialization', 'init': 'xavier', 'seed': 1},
]
topic_results = []
for exp_cfg in experiment_cfgs:
    result = run_experiment(exp_cfg, data)
    record_result(result); topic_results.append(result)
    delta = result['summary']['val_macro_f1'] - base_f1_mean
    print(f"  delta vs baseline mean={delta:+.4f}; exceeds 2std? {abs(delta) > noise_2sigma}")

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
baseline_reference = [baseline_results[0]]
for group, filename in [('optimizer', 'compare_optimizer.png'), ('dropout', 'compare_dropout.png'),
                        ('clipping', 'compare_clipping.png'), ('initialization', 'compare_initialization.png')]:
    group_results = baseline_reference + [r for r in topic_results if r['cfg']['group'] == group]
    plot_compare(group_results, 'val_macro_f1', f"{OUT_DIR}/figures/{filename}", f"{group}: validation macro-F1")
plot_compare(baseline_results, 'val_macro_f1', f"{OUT_DIR}/figures/compare_baseline_seeds.png", 'Baseline seed variation')

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# Final model is selected solely by validation macro-F1 from the completed experiments.
result_final = max(all_results, key=lambda r: r['summary']['val_macro_f1'])
cfg_final = result_final['cfg']
print('Final validation-selected config:', cfg_final['exp_id'], cfg_final)
final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
subprocess.run([sys.executable, 'scripts/evaluate.py', '--pred', f"{OUT_ABS}/predictions_eval.csv", '--out', f"{OUT_ABS}/eval_result.json"], cwd=REPO_ROOT, check=True, env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
# Eval is also run once for a representative baseline seed, never for configuration selection.
baseline_eval_result = baseline_results[0]
final_eval(baseline_eval_result['cfg'], baseline_eval_result, data, f"{OUT_DIR}/predictions_baseline.csv")
subprocess.run([sys.executable, 'scripts/evaluate.py', '--pred', f"{OUT_ABS}/predictions_baseline.csv", '--out', f"{OUT_ABS}/eval_baseline.json"], cwd=REPO_ROOT, check=True, env={**os.environ, 'PYTHONIOENCODING': 'utf-8'})
with open(f"{OUT_DIR}/eval_result.json") as handle: final_scores = json.load(handle)
with open(f"{OUT_DIR}/eval_baseline.json") as handle: baseline_scores = json.load(handle)
print('Final eval:', final_scores['accuracy'], final_scores['macro_f1'])
print('Baseline eval:', baseline_scores['accuracy'], baseline_scores['macro_f1'])

In [ ]:
import pandas as pd
per_class = pd.DataFrame(final_scores['per_class'])
display(per_class)
cm = np.asarray(final_scores['confusion_matrix'])
hardest = per_class.loc[per_class['f1'].idxmin()]
row = cm[int(hardest['cls'])].copy(); row[int(hardest['cls'])] = 0
confused_with = int(row.argmax())
print(f"Hardest class: {int(hardest['cls'])}; F1={hardest['f1']:.4f}; most often confused with class {confused_with} ({int(row.max())} samples).")
fig, ax = plt.subplots(figsize=(7, 6)); image = ax.imshow(cm, cmap='Blues'); fig.colorbar(image, ax=ax)
ax.set(xticks=np.arange(7), yticks=np.arange(7), xlabel='Predicted class', ylabel='True class', title='Final eval confusion matrix')
for i in range(7):
    for j in range(7): ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=7, color='white' if cm[i, j] > cm.max()/2 else 'black')
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/figures/confusion_matrix_eval.png", dpi=180); plt.show()

**Phân tích lỗi:** Lớp 5 khó nhất (F1=0.4926), chủ yếu bị nhầm thành lớp 2 (1,236 mẫu). Lớp 4 có recall thấp 0.4007 và thường bị nhầm thành lớp 1. Các lớp này ít mẫu hơn lớp 1; weighted loss hoặc sampling cân bằng là bước thử tiếp theo trên validation.

In [ ]:
results = load_results(f"{OUT_DIR}/results")
eval_by_id = {baseline_eval_result['cfg']['exp_id']: baseline_scores, cfg_final['exp_id']: final_scores}
rows = [to_row(result, eval_by_id.get(result['cfg']['exp_id']),
               notes='Selected by validation only.' if result['cfg']['exp_id'] == cfg_final['exp_id'] else '') for result in results]
write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
from openpyxl import load_workbook
workbook = load_workbook(f"{OUT_DIR}/experiments.xlsx")
seed_sheet = workbook['Seeds']; seed_sheet.delete_rows(2, seed_sheet.max_row)
for index, result in enumerate(baseline_results, start=2):
    summary = result['summary']; seed_sheet.cell(index, 1, result['cfg']['exp_id']); seed_sheet.cell(index, 2, result['cfg']['seed'])
    seed_sheet.cell(index, 3, summary['val_acc']); seed_sheet.cell(index, 4, summary['val_macro_f1'])
summary_sheet = workbook['Summary']
summary_sheet['A2'] = 'Baseline seed mean ± std (val macro-F1)'
summary_sheet['B2'] = f'{base_f1_mean:.4f} ± {base_f1_std:.4f}; 2σ={noise_2sigma:.4f}'
summary_sheet['A3'] = 'Final validation-selected experiment'
summary_sheet['B3'] = cfg_final['exp_id']
summary_sheet['A4'] = 'Final eval macro-F1 / accuracy'
summary_sheet['B4'] = f"{final_scores['macro_f1']:.4f} / {final_scores['accuracy']:.4f}"
workbook.save(f"{OUT_DIR}/experiments.xlsx")
print(f"Wrote {len(rows)} experiment rows to {OUT_DIR}/experiments.xlsx")